In [1]:
import os
for r, dirs, files in os.walk("/kaggle/input"):
    depth = r.count("/") - 2
    if depth <= 3:
        print(r, "| folders:", dirs[:6], "| files:", len(files))

/kaggle/input | folders: ['datasets'] | files: 0
/kaggle/input/datasets | folders: ['sabahesaraki'] | files: 0
/kaggle/input/datasets/sabahesaraki | folders: ['2017-2017'] | files: 0
/kaggle/input/datasets/sabahesaraki/2017-2017 | folders: ['annotations_trainval2017', 'image_info_test2017', 'image_info_unlabeled2017', 'test2017', 'train2017', 'unlabeled2017'] | files: 0


In [4]:
import os
root = "/kaggle/input/datasets/sabahesaraki/2017-2017"

print(os.listdir(root))

ann = f"{root}/annotations_trainval2017"
for r, d, f in os.walk(ann):
    print(r, "|", f[:10])
    break

for r, d, f in os.walk(ann):
    print(r, "->", [x for x in f if x.endswith(".json")])

['annotations_trainval2017', 'image_info_test2017', 'image_info_unlabeled2017', 'test2017', 'train2017', 'unlabeled2017', 'val2017']
/kaggle/input/datasets/sabahesaraki/2017-2017/annotations_trainval2017 | []
/kaggle/input/datasets/sabahesaraki/2017-2017/annotations_trainval2017 -> []
/kaggle/input/datasets/sabahesaraki/2017-2017/annotations_trainval2017/annotations -> ['captions_train2017.json', 'captions_val2017.json', 'instances_train2017.json', 'instances_val2017.json', 'person_keypoints_train2017.json', 'person_keypoints_val2017.json']


In [5]:
for name in ["train2017", "val2017"]:
    p = f"{root}/{name}"
    if os.path.exists(p):
        n = 0
        for r, d, f in os.walk(p):
            n += len(f)
        print(name, n)
    else:
        print(name, "NOT FOUND")
        

train2017 118287
val2017 5000


In [7]:
import os, json, collections

root = "/kaggle/input/datasets/sabahesaraki/2017-2017"
ann  = f"{root}/annotations_trainval2017/annotations"

def find_img_dir(name):
    p = f"{root}/{name}"
    sub = f"{p}/{name}"
    return sub if os.path.isdir(sub) else p

img_dir = {s: find_img_dir(s) for s in ["train2017", "val2017"]}
print(img_dir)
print(os.listdir(img_dir["train2017"])[:3])

{'train2017': '/kaggle/input/datasets/sabahesaraki/2017-2017/train2017/train2017', 'val2017': '/kaggle/input/datasets/sabahesaraki/2017-2017/val2017/val2017'}
['000000000009.jpg', '000000000025.jpg', '000000000030.jpg']


In [8]:
def audit(split):
    d = json.load(open(f"{ann}/captions_{split}.json"))
    imgs, anns = d["images"], d["annotations"]
    ids = [i["id"] for i in imgs]

    caps = collections.defaultdict(list)
    for a in anns:
        caps[a["image_id"]].append(a["caption"])

    on_disk = set(os.listdir(img_dir[split]))
    report = {
        "split": split,
        "num_images": len(ids),
        "num_captions": len(anns),
        "duplicate_image_ids": len(ids) - len(set(ids)),
        "images_missing_on_disk": sum(1 for i in imgs if i["file_name"] not in on_disk),
        "orphan_captions": len(set(caps) - set(ids)),
        "empty_captions": sum(1 for a in anns if not a["caption"].strip()),
        "images_with_less_than_5": sum(1 for i in ids if len(caps[i]) < 5),
        "captions_per_image_distribution": dict(collections.Counter(len(caps[i]) for i in ids)),
    }
    return report

reports = [audit("train2017"), audit("val2017")]
for r in reports:
    print(json.dumps(r, indent=2))

os.makedirs("/kaggle/working/reports", exist_ok=True)
json.dump(reports, open("/kaggle/working/reports/audit_report.json", "w"), indent=2)

{
  "split": "train2017",
  "num_images": 118287,
  "num_captions": 591753,
  "duplicate_image_ids": 0,
  "images_missing_on_disk": 0,
  "orphan_captions": 0,
  "empty_captions": 0,
  "images_with_less_than_5": 0,
  "captions_per_image_distribution": {
    "5": 117972,
    "6": 312,
    "7": 3
  }
}
{
  "split": "val2017",
  "num_images": 5000,
  "num_captions": 25014,
  "duplicate_image_ids": 0,
  "images_missing_on_disk": 0,
  "orphan_captions": 0,
  "empty_captions": 0,
  "images_with_less_than_5": 0,
  "captions_per_image_distribution": {
    "5": 4987,
    "6": 12,
    "7": 1
  }
}


In [2]:
import os, json, collections

root = "/kaggle/input/datasets/sabahesaraki/2017-2017"
ann  = f"{root}/annotations_trainval2017/annotations"

def find_img_dir(name):
    p = f"{root}/{name}"
    sub = f"{p}/{name}"
    return sub if os.path.isdir(sub) else p

img_dir = {s: find_img_dir(s) for s in ["train2017", "val2017"]}
print(img_dir)

{'train2017': '/kaggle/input/datasets/sabahesaraki/2017-2017/train2017/train2017', 'val2017': '/kaggle/input/datasets/sabahesaraki/2017-2017/val2017/val2017'}


In [3]:
def audit(split):
    d = json.load(open(f"{ann}/captions_{split}.json"))
    imgs, anns = d["images"], d["annotations"]
    ids = [i["id"] for i in imgs]

    caps = collections.defaultdict(list)
    for a in anns:
        caps[a["image_id"]].append(a["caption"])

    on_disk = set(os.listdir(img_dir[split]))
    return {
        "split": split,
        "num_images": len(ids),
        "num_captions": len(anns),
        "duplicate_image_ids": len(ids) - len(set(ids)),
        "images_missing_on_disk": sum(1 for i in imgs if i["file_name"] not in on_disk),
        "orphan_captions": len(set(caps) - set(ids)),
        "empty_captions": sum(1 for a in anns if not a["caption"].strip()),
        "images_with_less_than_5": sum(1 for i in ids if len(caps[i]) < 5),
        "captions_per_image_distribution": dict(collections.Counter(len(caps[i]) for i in ids)),
    }

reports = [audit("train2017"), audit("val2017")]
for r in reports:
    print(json.dumps(r, indent=2))

os.makedirs("/kaggle/working/reports", exist_ok=True)
json.dump(reports, open("/kaggle/working/reports/audit_report.json", "w"), indent=2)

{
  "split": "train2017",
  "num_images": 118287,
  "num_captions": 591753,
  "duplicate_image_ids": 0,
  "images_missing_on_disk": 0,
  "orphan_captions": 0,
  "empty_captions": 0,
  "images_with_less_than_5": 0,
  "captions_per_image_distribution": {
    "5": 117972,
    "6": 312,
    "7": 3
  }
}
{
  "split": "val2017",
  "num_images": 5000,
  "num_captions": 25014,
  "duplicate_image_ids": 0,
  "images_missing_on_disk": 0,
  "orphan_captions": 0,
  "empty_captions": 0,
  "images_with_less_than_5": 0,
  "captions_per_image_distribution": {
    "5": 4987,
    "6": 12,
    "7": 1
  }
}


In [4]:
import random, hashlib
from PIL import Image, ImageOps
from tqdm import tqdm

SEED = 42
cap = {s: json.load(open(f"{ann}/captions_{s}.json")) for s in ["train2017", "val2017"]}

id2file, id2caps = {}, {}
for s in cap:
    id2file[s] = {i["id"]: i["file_name"] for i in cap[s]["images"]}
    tmp = collections.defaultdict(list)
    for a in sorted(cap[s]["annotations"], key=lambda a: a["id"]):
        tmp[a["image_id"]].append(a["caption"].strip())
    id2caps[s] = {k: v[:5] for k, v in tmp.items() if len(v) >= 5}

train_pool = list(id2caps["train2017"].keys())
print("train pool:", len(train_pool))

train pool: 118287


In [5]:
def check_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            im.load()
            ImageOps.exif_transpose(im).convert("RGB")
        return True, ""
    except Exception as e:
        return False, str(e)

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

rng = random.Random(SEED)

def pick(pool, n, split, seen):
    pool = sorted(pool)
    rng.shuffle(pool)
    chosen, rejected = [], []
    for i in tqdm(pool, desc=f"{split} n={n}"):
        if len(chosen) == n:
            break
        p = f"{img_dir[split]}/{id2file[split][i]}"
        ok, info = check_image(p)
        if not ok:
            rejected.append({"id": i, "reason": "unreadable", "detail": info})
            continue
        h = md5(p)
        if h in seen:
            rejected.append({"id": i, "reason": "duplicate_content"})
            continue
        seen[h] = i
        chosen.append(i)
    return chosen, rejected

In [6]:
seen_train, seen_val = {}, {}

chosen, rej_train = pick(train_pool, 10100, "train2017", seen_train)
kb_ids, val_ids = chosen[:10000], chosen[10000:]

inst = json.load(open(f"{ann}/instances_val2017.json"))
cnt = collections.Counter(a["image_id"] for a in inst["annotations"])
bins = {"simple": [], "medium": [], "complex": []}
for i, n in cnt.items():
    if i in id2caps["val2017"]:
        bins["simple" if n <= 3 else "medium" if n <= 7 else "complex"].append(i)
print({k: len(v) for k, v in bins.items()})

quota = {"simple": 300, "medium": 400, "complex": 300}
test_ids, rej_test, test_bin = [], [], {}
for k, q in quota.items():
    c, r = pick(bins[k], q, "val2017", seen_val)
    test_ids += c
    rej_test += r
    for i in c:
        test_bin[i] = k

print(len(kb_ids), len(val_ids), len(test_ids))
print("rejected train:", len(rej_train), "| rejected test:", len(rej_test))

train2017 n=10100:   9%|▊         | 10100/118287 [02:53<31:00, 58.14it/s]


{'simple': 2062, 'medium': 1192, 'complex': 1698}


val2017 n=300:  18%|█▊        | 300/1698 [00:04<00:21, 63.85it/s]

10000 100 1000
rejected train: 0 | rejected test: 0


In [8]:
out = "/kaggle/working/splits"
os.makedirs(out, exist_ok=True)

def manifest(ids, split, extra=None):
    return [{"image_id": i, "file_name": id2file[split][i], "captions": id2caps[split][i],
             **({"stratum": extra[i]} if extra else {})} for i in sorted(ids)]

for name, ids in [("kb_ids", kb_ids), ("val_ids", val_ids), ("test_ids", test_ids)]:
    json.dump(sorted(ids), open(f"{out}/{name}.json", "w"))

json.dump(manifest(kb_ids, "train2017"),  open(f"{out}/kb_manifest.json", "w"))
json.dump(manifest(val_ids, "train2017"), open(f"{out}/val_manifest.json", "w"))
json.dump(manifest(test_ids, "val2017", test_bin), open(f"{out}/test_manifest.json", "w"))

json.dump({"seed": SEED, "quota_test": quota, "caption_policy": "first 5 captions by annotation id",
           "rejected_train": rej_train, "rejected_test": rej_test},
          open(f"{out}/split_meta.json", "w"), indent=2)
print("saved")

saved


In [9]:
kb, va, te = set(kb_ids), set(val_ids), set(test_ids)
kb_hashes = {h for h, i in seen_train.items() if i in kb}
val_hashes = {h for h, i in seen_train.items() if i in va}

leak = {
    "sizes": {"kb": len(kb), "val": len(va), "test": len(te)},
    "kb_val_overlap": len(kb & va),
    "kb_test_id_overlap": len(kb & te),
    "val_test_id_overlap": len(va & te),
    "kb_test_content_overlap": len(set(seen_train) & set(seen_val)),
    "val_content_in_kb": len(val_hashes & kb_hashes),
    "rejected_unreadable": sum(r["reason"] == "unreadable" for r in rej_train + rej_test),
    "rejected_duplicates": sum(r["reason"] == "duplicate_content" for r in rej_train + rej_test),
}
assert leak["sizes"] == {"kb": 10000, "val": 100, "test": 1000}
assert leak["kb_val_overlap"] == leak["kb_test_id_overlap"] == leak["val_test_id_overlap"] == 0
print(json.dumps(leak, indent=2))

rep = json.load(open("/kaggle/working/reports/audit_report.json"))
rep.append({"selected_images_check": leak})
json.dump(rep, open("/kaggle/working/reports/audit_report.json", "w"), indent=2)
json.dump(leak, open("/kaggle/working/reports/leakage_audit.json", "w"), indent=2)

{
  "sizes": {
    "kb": 10000,
    "val": 100,
    "test": 1000
  },
  "kb_val_overlap": 0,
  "kb_test_id_overlap": 0,
  "val_test_id_overlap": 0,
  "kb_test_content_overlap": 0,
  "val_content_in_kb": 0,
  "rejected_unreadable": 0,
  "rejected_duplicates": 0
}
